# NOTEBOOK 06 — Conteo, Bucles Anidados y Ordenamiento

**Semana 09 — Fundamentos de Python** | Docente: Ing. Andrés Mena Abarca | UMCA

**Alumno:** ODEL LEAL HERNANDEZ

Temas: `dict` como contador · `collections.Counter` · bucles anidados · `sorted()` con `key=lambda`


---

## ¿Qué trabajamos hoy?

En S08 aprendimos a trabajar con listas de diccionarios: acceder a campos, filtrar con `for + if` y contar con un acumulador.
Hoy damos el siguiente paso: **contar de forma eficiente**, **recorrer datos dentro de datos** y **ordenar resultados**.

| Fase | Tema | Tiempo |
|------|------|--------|
| Ritual de inicio | Conexión con S08 | 5 min |
| Setup | Carga del archivo JSON | 8 min |
| Fase 0 | El problema del conteo (demo en vivo) | 20 min |
| Fase 1 | Diccionario como contador · `Counter` | 30 min |
| Fase 2 | Bucles anidados sobre `sintomas` | 35 min |
| Fase 3 | `sorted()` con `key=lambda` | 30 min |
| Fase 4 | Integrador libre | 25 min |
| Cierre | Reflexión + anticipo S10 | 7 min |
| **Total** | | **160 min** |

---

## Objetivos de aprendizaje

Al terminar esta sesión seremos capaces de:

- Usar `dict.get(clave, 0) + 1` para contar cualquier campo de un dataset en un solo recorrido
- Distinguir cuándo usar `Counter` y cuándo el patrón manual
- Escribir bucles `for` anidados para recorrer datos dentro de datos
- Ordenar resultados por valor usando `sorted()` con `key=lambda`
- Producir visualizaciones de texto (barras ASCII) sin librerías externas

---
## Ritual de inicio — Conectemos con S08

> ✋ **Checkpoint oral — El docente conduce esta sección antes de abrir el notebook.**

1. ¿Cómo accedemos al campo `enfermedad` de un paciente almacenado en un diccionario?
2. ¿Qué hace el patrón `for p in pacientes: if p["enfermedad"] == "gripe": contador += 1`?
3. Si queremos contar **todas** las enfermedades al mismo tiempo con ese patrón, ¿qué problema encontramos?
4. **Predicción:** si tenemos 1 000 000 de pacientes y 40 enfermedades distintas, ¿cuántos pacientes en promedio tiene cada enfermedad?

---
## Setup — Carga de datos

⏱ 8 minutos

> **Nota sobre el import:** importamos `Counter` de `collections`.
> Lo usaremos en **Fase 1** — por ahora ignoramos esa línea.

**Campos del dataset `clinica_s09.json`:**

| Campo | Tipo | Descripción |
|-------|------|-------------|
| `carnet` | int | Número de identificación |
| `nombre` | str | Nombre completo |
| `edad` | int | Edad en años (5–90) |
| `genero` | str | `"M"` o `"F"` |
| `provincia` | str | Provincia de residencia |
| `canton` | str | Cantón de residencia |
| `visitas` | int | Número de visitas (1–30) |
| `enfermedad` | str | Diagnóstico principal |
| `medicamento` | str | Medicamento recetado |
| `activo` | bool | Si el expediente está activo |
| `sintomas` | list | Lista de 3 síntomas |

In [3]:
import json
import os
from collections import Counter  # se explica en Fase 1 — por ahora ignorar

ARCHIVO = "clinica_s09.json"

if not os.path.exists(ARCHIVO):
    print(f"Archivo '{ARCHIVO}' no encontrado en: {os.getcwd()}")
    print("Ejecuta primero: python generar_clinica_s09.py")
    pacientes = []
else:
    with open(ARCHIVO, "r", encoding="utf-8") as archivo:
        pacientes = json.load(archivo)
    print(f"Cargados {len(pacientes):,} pacientes correctamente.")
    print(f"Campos disponibles: {list(pacientes[0].keys())}")

Cargados 1,000,000 pacientes correctamente.
Campos disponibles: ['carnet', 'nombre', 'edad', 'genero', 'provincia', 'canton', 'visitas', 'enfermedad', 'medicamento', 'activo', 'sintomas']


In [4]:
# ── Exploración inicial del dataset ───────────────────────────────────────
print(f"Total de registros : {len(pacientes):,}")
print(f"Campos disponibles : {list(pacientes[0].keys())}")
print()
print("── Primer paciente de ejemplo ──")
for campo, valor in pacientes[0].items():
    print(f"  {campo:<15}: {valor}")

Total de registros : 1,000,000
Campos disponibles : ['carnet', 'nombre', 'edad', 'genero', 'provincia', 'canton', 'visitas', 'enfermedad', 'medicamento', 'activo', 'sintomas']

── Primer paciente de ejemplo ──
  carnet         : 890779946
  nombre         : Carlos Romero Núñez
  edad           : 18
  genero         : M
  provincia      : Alajuela
  canton         : San Ramón
  visitas        : 22
  enfermedad     : alzheimer
  medicamento    : losartán
  activo         : True
  sintomas       : ['náuseas', 'fatiga', 'dolor muscular']


---
## Fase 0 — El problema del conteo

> **Objetivo:** ver en vivo por qué el patrón de listas paralelas no escala. Sentir el problema antes de aprender la solución.

⏱ 20 minutos

> **¿Por qué importa?** En análisis de datos siempre necesitamos contar: enfermedades, productos, palabras, clics.
> La estructura que elijamos para almacenar esos conteos determina si nuestro código funciona con 100 registros o con 1 000 000.

In [5]:
# ── Celda 0.A — Repaso S08: contar pacientes con gripe ────────────────────
conteo_gripe = 0
for p in pacientes:
    if p["enfermedad"] == "gripe":
        conteo_gripe += 1
print(f"Pacientes con gripe: {conteo_gripe}")

Pacientes con gripe: 24898


> **Predicción antes de ver el código lento:**
> Si quisiéramos contar las 40 enfermedades usando el patrón anterior, necesitaríamos 40 bloques `if` como ese.
> ¿Se te ocurre una estructura que almacene todos los conteos a la vez en un solo recorrido?

### Celda 0.B — El problema con listas paralelas

¿Cómo contaríamos **todas** las enfermedades con el patrón de S08?
Una solución posible es usar dos listas paralelas. Observa el código y luego medimos cuánto tarda.

> **Predicción:** ¿crees que tardará más o menos que el diccionario (Celda 0.C)?
> Escribe mentalmente tu respuesta antes de ejecutar.

In [6]:
%%time
# ── Celda 0.B — Listas paralelas sobre 1 000 000 pacientes ────────────────
muestra = pacientes  # dataset completo — misma condición que 0.C

nombres_enfermedades = []
conteos = []
for p in muestra:
    enfermedad = p["enfermedad"]
    if enfermedad in nombres_enfermedades:
        posicion = nombres_enfermedades.index(enfermedad)  # recorre la lista cada vez
        conteos[posicion] += 1
    else:
        nombres_enfermedades.append(enfermedad)
        conteos.append(1)

print(f"Enfermedades encontradas: {len(nombres_enfermedades)}")

Enfermedades encontradas: 40
CPU times: total: 891 ms
Wall time: 895 ms


### Celda 0.C — El mismo conteo con diccionario

Ahora contamos **todos** los 1 000 000 de pacientes usando un diccionario. Observa el tiempo.

In [7]:
%%time
# ── Celda 0.C — Diccionario sobre el 1 000 000 de pacientes ───────────────────
conteo_dict = {}
for p in pacientes:
    e = p["enfermedad"]
    conteo_dict[e] = conteo_dict.get(e, 0) + 1

print(f"Enfermedades encontradas: {len(conteo_dict)}")

Enfermedades encontradas: 40
CPU times: total: 328 ms
Wall time: 303 ms


### ¿Qué acabamos de ver?

| Enfoque | Datos procesados | Tiempo aprox. |
|---------|-----------------|---------------|
| Listas paralelas con `.index()` | 1 000 000 pacientes | ~0.75 s |
| Diccionario con `.get()` | 1 000 000 pacientes | ~0.37 s |

> **Nota:** con enfermedades (40 valores únicos) la diferencia es de ~2×.
> Con campos de alta cardinalidad (ej: nombres de pacientes, miles de valores únicos),
> la lista crece sin límite y la brecha se vuelve **exponencial**.

> ✋ **Checkpoint oral:**
> 1. ¿Qué estructura de datos usamos en S08 para acceder por nombre?
> 2. ¿Por qué `.index()` se vuelve más lento a medida que la lista crece?
> 3. ¿Podrías usar un diccionario para guardar los conteos directamente?

---
## Fase 1 — Diccionario como contador

> **Objetivo:** dominar el patrón `.get(clave, 0) + 1`, luego conocer `Counter` como alternativa profesional.

⏱ 30 minutos

### Concepto — ¿Por qué `.get()` y no `dict["clave"]`?

Cuando intentamos acceder a una clave que no existe, Python lanza un error:

```python
conteo = {}
conteo["gripe"] += 1  # ❌ KeyError: 'gripe' — la clave no existe todavía
```

El método `.get(clave, valor_por_defecto)` resuelve esto elegantemente:

```python
conteo["gripe"] = conteo.get("gripe", 0) + 1  # ✓ retorna 0 si no existe
```

**Leemos la línea en voz alta:** *"El nuevo conteo de gripe es el conteo actual de gripe (o cero si no existía) más uno."*

Este patrón de **una sola línea** reemplaza todo el bloque de listas paralelas que vimos en Fase 0.

### Desafío 1.A — Conteo con diccionario

⏱ 8 minutos | Individual

Recorre los **1 000 000 de pacientes** y cuenta cuántos hay por enfermedad.
Guarda el resultado en un diccionario llamado `conteo_enfermedades`.

**Estructura esperada del resultado:**
```python
{
    "gripe": 24350,
    "diabetes": 25100,
    "migraña": 24890,
    ...  # 40 enfermedades en total
}
```

**Patrón a usar:** `conteo[clave] = conteo.get(clave, 0) + 1`

In [8]:
conteo_enfermedades = {}

for  p in pacientes:
    e = p["enfermedad"]
    e = e.lower().strip()
    conteo_enfermedades[e] = conteo_enfermedades.get(e, 0) + 1

print(f" El total de Enfermedades encontradas son: {len(conteo_enfermedades)}")
print("-" * 50)
print(" Enfermedad y cantidad de pacientes")
print("-" * 50)

for enfermedad, cantidad in conteo_enfermedades.items():
    print(f"✅ {enfermedad:<25}: {cantidad:,} pacientes")
    


 El total de Enfermedades encontradas son: 40
--------------------------------------------------
 Enfermedad y cantidad de pacientes
--------------------------------------------------
✅ alzheimer                : 24,767 pacientes
✅ anemia                   : 25,009 pacientes
✅ alergia                  : 25,472 pacientes
✅ lumbalgia                : 24,914 pacientes
✅ conjuntivitis            : 25,105 pacientes
✅ dermatitis               : 24,615 pacientes
✅ artritis                 : 24,856 pacientes
✅ epilepsia                : 25,083 pacientes
✅ parkinson                : 25,267 pacientes
✅ migraña                  : 24,956 pacientes
✅ hepatitis                : 25,074 pacientes
✅ sinusitis                : 24,606 pacientes
✅ ansiedad generalizada    : 25,256 pacientes
✅ lupus                    : 24,943 pacientes
✅ psoriasis                : 25,131 pacientes
✅ insuficiencia renal      : 25,127 pacientes
✅ gripe                    : 24,898 pacientes
✅ fibromialgia             : 25,09

In [9]:
try:
    assert isinstance(conteo_enfermedades, dict), "conteo_enfermedades debe ser un dict"
    assert len(conteo_enfermedades) == 40, f"Enfermedades distintas: esperado 40, obtenido {len(conteo_enfermedades)}\n  Pista: ¿Recorres todos los pacientes? ¿Inicializaste el dict fuera del for?"
    assert sum(conteo_enfermedades.values()) == 1_000_000, f"Total de pacientes: esperado 1 000 000, obtenido {sum(conteo_enfermedades.values())}\n  Pista: ¿Estás usando todos los pacientes o solo una muestra?"
    print("✓ Desafío 1.A correcto")
except NameError as e:
    print(f"Variable no definida: {e}")
    print("  ¿Ejecutaste la celda del desafío antes de esta?")
except AssertionError as e:
    print(f"✗ {e}")

✓ Desafío 1.A correcto


> ⚠️ **Error frecuente — El dict dentro del `for`**
>
> ```python
> # ❌ INCORRECTO — se reinicia en cada paciente
> for p in pacientes:
>     conteo_enfermedades = {}          # borra todo lo acumulado
>     e = p["enfermedad"]
>     conteo_enfermedades[e] = conteo_enfermedades.get(e, 0) + 1
>
> # ✓ CORRECTO — se inicializa una sola vez antes del for
> conteo_enfermedades = {}
> for p in pacientes:
>     e = p["enfermedad"]
>     conteo_enfermedades[e] = conteo_enfermedades.get(e, 0) + 1
> ```
>
> Si tu verificación da 40 enfermedades pero cada una con conteo `1`, probablemente cometiste este error.

### Reflexión — Antes de conocer `Counter`

Mira tu código de 1.A y responde:

1. ¿Cuántas líneas de código necesitaste para contar todas las enfermedades?
2. Si mañana necesitas contar por `provincia` en lugar de `enfermedad`, ¿qué cambiarías?
3. ¿Qué pasaría si el campo `enfermedad` tuviera errores de escritura (ej: `"Gripe"` vs `"gripe"`)?

> Guarda mentalmente tus respuestas. Las compararemos con lo que viene a continuación.

In [10]:
# ── collections.Counter hace lo mismo que 1.A en una línea ────────────────
# Counter recibe cualquier iterable y cuenta sus elementos automáticamente
conteo_con_counter = Counter(p["enfermedad"] for p in pacientes)
print(conteo_con_counter.most_common(5))

[('alergia', 25472), ('otitis', 25297), ('esclerosis múltiple', 25273), ('parkinson', 25267), ('ansiedad generalizada', 25256)]


> 🔗 **Conexión con Pandas (anticipo S10)**
>
> Lo que acabamos de hacer con `Counter` en una línea:
> ```python
> Counter(p["enfermedad"] for p in pacientes)
> ```
> En Pandas se escribirá así:
> ```python
> df["enfermedad"].value_counts()
> ```
> Misma idea, mismos resultados — Pandas solo lo hace más rápido y con más opciones de formato.

### Desafío 1.C — Counter de medicamentos

⏱ 7 minutos | Individual

Usa `Counter` para contar los **medicamentos** de todos los pacientes.
Guarda el resultado en `conteo_medicamentos`.

**Patrón de Counter con generador:**
```python
conteo_medicamentos = Counter(p["campo"] for p in pacientes)
```

Luego imprime:
- Cantidad de medicamentos **distintos**: `len(conteo_medicamentos)`
- El **más recetado** y cuántas veces: `conteo_medicamentos.most_common(1)`

In [11]:
%%time
# Escribe tu código aquí
conteo_medicamentos = Counter(p["medicamento"] for p in pacientes)
print(f"El total de medicamentos es:👉 {len(conteo_medicamentos)}")
print("-" * 50)
mas_recetado, cantidad = conteo_medicamentos.most_common(1)[0]
print(f"El medicamente más recetado es: 👉 {mas_recetado} y {cantidad:,} veces.")
print("-" * 50)
print("Los medicamentos más comunes 'Top 3' son:👇")
for medicamento, cantidad in conteo_medicamentos.most_common(3):
    print(f"✅ {medicamento:<25}: {cantidad:,} pacientes")

print("-"* 50)

El total de medicamentos es:👉 35
--------------------------------------------------
El medicamente más recetado es: 👉 azitromicina y 28,852 veces.
--------------------------------------------------
Los medicamentos más comunes 'Top 3' son:👇
✅ azitromicina             : 28,852 pacientes
✅ ranitidina               : 28,850 pacientes
✅ amoxicilina              : 28,793 pacientes
--------------------------------------------------
CPU times: total: 172 ms
Wall time: 181 ms


In [12]:
try:
    assert sum(conteo_medicamentos.values()) == 1_000_000, f"Total de recetas: esperado 1 000 000, obtenido {sum(conteo_medicamentos.values())}\n  Pista: ¿Estás usando todos los pacientes? ¿Usaste Counter() con el campo correcto?"
    assert len(conteo_medicamentos) == 35, f"Medicamentos distintos: esperado 35, obtenido {len(conteo_medicamentos)}\n  Pista: ¿Usaste el campo 'medicamento' de cada paciente?"
    assert conteo_medicamentos.most_common(1)[0][0] == "azitromicina", f"Más recetado: esperado 'azitromicina', obtenido '{conteo_medicamentos.most_common(1)[0][0]}'\n  Pista: ¿Usaste Counter() con el campo 'medicamento' de todos los pacientes?"
    assert conteo_medicamentos.most_common(1)[0][1] == 28_852, f"Recetas de 'azitromicina': esperado 28 852, obtenido {conteo_medicamentos.most_common(1)[0][1]}\n  Pista: ¿Estás usando los 1 000 000 pacientes completos?"
    print("✓ Desafío 1.C correcto")
except NameError as e:
    print(f"Variable no definida: {e}")
    print("  ¿Ejecutaste la celda del desafío antes de esta?")
except AssertionError as e:
    print(f"✗ {e}")

✓ Desafío 1.C correcto


> 🤝 **Revisión en pareja — 5 min**
> 1. ¿Obtuvieron los mismos resultados en 1.A y 1.C?
> 2. ¿Cuántas líneas usaron en 1.A? ¿Y con Counter en 1.C?
> 3. ¿En qué situaciones preferirían el patrón manual sobre `Counter`?

> ✋ **Checkpoint oral:**
> 1. ¿Qué devuelve `dict.get("clave_inexistente", 0)`?
> 2. ¿Cuál es la diferencia entre el patrón manual y `Counter`?
> 3. ¿Cuándo preferirías usar `Counter` sobre el patrón manual?
>
> **Anticipo S10:** `df["enfermedad"].value_counts()` en Pandas hace lo mismo que Counter, pero opera sobre millones de filas en microsegundos usando vectorización.

---
## Fase 2 — Bucles anidados sobre `sintomas`

> **Objetivo:** escribir nuestro primer `for` dentro de `for` con un propósito real.

⏱ 35 minutos

### Concepto — ¿Qué es un bucle anidado?

Hasta ahora nuestros bucles recorren una lista plana: cada elemento es un valor simple.
Pero el campo `sintomas` de cada paciente **es a su vez una lista**.
Para llegar a cada síntoma individual necesitamos **dos `for`**: uno que abre el paciente, otro que recorre sus síntomas.

```python
# Analogía: una caja con sobres, cada sobre tiene tarjetas dentro
for paciente in pacientes:                # nivel 1 — abrimos cada paciente
    for sintoma in paciente["sintomas"]:  # nivel 2 — recorremos su lista interna
        print(sintoma)                    # dato individual: una tarjeta
```

**Regla práctica:** si el dato que necesitas está *dentro* de otro dato, necesitas un `for` por cada nivel de profundidad.

> **Conexión con S08:** el `for` externo es exactamente igual a los que usamos en S08.
> Lo único nuevo es que cada elemento contiene otra lista que también debemos recorrer.

In [13]:
# ── Celda 2.0 — Un solo paciente con sus síntomas ──────────────────────────
primer_paciente = pacientes[0]
print(f"Paciente: {primer_paciente['nombre']}")
print(f"Síntomas: {primer_paciente['sintomas']}")

# Para recorrer sus síntomas necesitamos otro for
for sintoma in primer_paciente["sintomas"]:
    print(f"  - {sintoma}")

Paciente: Carlos Romero Núñez
Síntomas: ['náuseas', 'fatiga', 'dolor muscular']
  - náuseas
  - fatiga
  - dolor muscular


> **Observa la estructura antes de continuar:**
> El campo `sintomas` es una **lista de 3 elementos**.
> Para acceder a cada síntoma individualmente necesitamos un segundo `for` que recorra esa lista.
> Esto es lo que veremos en los Desafíos 2.A y 2.B.

### Desafío 2.A — Imprimir síntomas de todos los pacientes

⏱ 5 minutos | Individual

Recorre los primeros **5 pacientes** e imprime su nombre y síntomas, uno por línea con sangría.
Este desafío te prepara para 2.B: primero visualiza la estructura, luego la contamos.

Usa **dos ciclos `for`**: el externo recorre pacientes, el interno recorre `p["sintomas"]`.

> Usa `pacientes[:5]` — con 1 000 000 de registros imprimir todo bloquearía el notebook.

In [14]:
# Escribe tu código aquí


for paciente in pacientes[:5]:
    # nombre del paciente
    print(f"\n Paciente: {paciente['nombre']}")
    print("-"*30)
    # Para recorrer sus síntomas necesitamos otro for
    for sintoma in paciente["sintomas"]:
        print(f"  - {sintoma}")
    




 Paciente: Carlos Romero Núñez
------------------------------
  - náuseas
  - fatiga
  - dolor muscular

 Paciente: Sergio Fernández Picado
------------------------------
  - confusión
  - fiebre
  - dificultad para tragar

 Paciente: Carolina Romero Vargas
------------------------------
  - depresión
  - vómitos
  - heridas que no sanan

 Paciente: Ernesto Pérez Madrigal
------------------------------
  - palpitaciones
  - dolor de pecho
  - debilidad muscular

 Paciente: Ernesto Romero Quesada
------------------------------
  - dificultad para respirar
  - pérdida de apetito
  - insomnio


### Desafío 2.B — Conteo total de síntomas

⏱ 10 minutos | Individual

Cada paciente tiene exactamente **3 síntomas**. Queremos saber cuántas veces aparece cada síntoma sumando a través de **todos** los pacientes.

> **Predicción:** con 1 000 000 de pacientes y 3 síntomas cada uno,
> ¿cuánto debe sumar el total de todos los conteos?

Usa un **diccionario contador** llamado `conteo_sintomas` y **dos `for` anidados**.

> ⚠️ **Error frecuente:** escribir `conteo_sintomas = {}` *dentro* del `for` externo
> reinicia el diccionario en cada paciente y borra todo lo acumulado.
> Inicialízalo **antes** del primer `for`.

In [15]:
%%time
# Escribe tu código aquí
conteo_sintomas = {}

for p in pacientes:
    for s in p["sintomas"]:
        conteo_sintomas[s] = conteo_sintomas.get(s,0) + 1

for sintoma, cantidad in conteo_sintomas.items():
    print(f"Síntoma: {sintoma} y {cantidad:,} veces.")


Síntoma: náuseas y 46,983 veces.
Síntoma: fatiga y 46,604 veces.
Síntoma: dolor muscular y 47,090 veces.
Síntoma: confusión y 47,126 veces.
Síntoma: fiebre y 46,605 veces.
Síntoma: dificultad para tragar y 46,798 veces.
Síntoma: depresión y 46,820 veces.
Síntoma: vómitos y 46,763 veces.
Síntoma: heridas que no sanan y 47,033 veces.
Síntoma: palpitaciones y 46,676 veces.
Síntoma: dolor de pecho y 47,220 veces.
Síntoma: debilidad muscular y 46,640 veces.
Síntoma: dificultad para respirar y 46,561 veces.
Síntoma: pérdida de apetito y 47,176 veces.
Síntoma: insomnio y 46,968 veces.
Síntoma: ictericia y 46,752 veces.
Síntoma: sensibilidad al sonido y 47,079 veces.
Síntoma: irritabilidad y 47,150 veces.
Síntoma: desmayos y 47,470 veces.
Síntoma: mareos y 47,151 veces.
Síntoma: rigidez matutina y 46,742 veces.
Síntoma: moretones y 47,250 veces.
Síntoma: picazón y 46,801 veces.
Síntoma: micción frecuente y 46,901 veces.
Síntoma: dolor de garganta y 47,095 veces.
Síntoma: mal aliento y 46,596 v

In [16]:
try:
    assert isinstance(conteo_sintomas, dict), "conteo_sintomas debe ser un dict"
    assert sum(conteo_sintomas.values()) == 3_000_000, f"Total de ocurrencias: esperado 3 000 000, obtenido {sum(conteo_sintomas.values())}\n  Pista: 1 000 000 pacientes × 3 síntomas = 3 000 000. ¿Tienes dos for anidados? ¿Recorres p['sintomas'] en el for interno?"
    assert len(conteo_sintomas) == 64, f"Síntomas distintos: esperado 64, obtenido {len(conteo_sintomas)}\n  Pista: ¿Estás contando síntomas individuales? ¿El for interno recorre p['sintomas']?"
    print("✓ Desafío 2.B correcto")
except NameError as e:
    print(f"Variable no definida: {e}")
    print("  ¿Ejecutaste la celda del desafío antes de esta?")
except AssertionError as e:
    print(f"✗ {e}")

✓ Desafío 2.B correcto


### Desafío 2.C — Síntomas únicos de San José

⏱ 10 minutos | Individual

Construye una lista con los síntomas únicos (sin repetidos) de los pacientes de **San José**.
Guarda el resultado en `sintomas_san_jose`.

**Estructura del algoritmo:**
1. `for p in pacientes` — recorre todos los pacientes
2. `if p["provincia"] == "San José"` — filtra por provincia
3. `for sintoma in p["sintomas"]` — recorre su lista de síntomas
4. `if sintoma not in sintomas_san_jose` — agrega solo si no está ya

> El dataset tiene 64 síntomas posibles. ¿Cuántos esperas encontrar en San José?

In [17]:

# Escribe tu código aquí
# sintomas_san_jose = Counter(s for p in pacientes if p["provincia"].lower().strip() == "San José" or p["provincia"].lower().strip() == "san jose" for s in p["sintomas"])
# print(f"Total de síntomas en San José: {sum(sintomas_san_jose.values()):,}")
# print(f"Síntomas distintos en San José: {len(sintomas_san_jose)}")
# print("Síntomas más comunes en San José:")
# for sintoma, cantidad in sintomas_san_jose.most_common(5):
#     print(f"✅ {sintoma:<25}: {cantidad:,} pacientes")
# print("-" * 50)

sintomas_san_jose = []
for p in pacientes:
    if p["provincia"].strip().lower() in ("san josé", "san jose"):
        for s in p["sintomas"]:
            if s not in sintomas_san_jose:
                sintomas_san_jose.append(s)

print(f"Total de síntomas en San José: {len(sintomas_san_jose):,}")
print("Síntomas encontrados en San José:")
for sintoma in sintomas_san_jose:
    print(f"✅ {sintoma}")  




Total de síntomas en San José: 64
Síntomas encontrados en San José:
✅ confusión
✅ fiebre
✅ dificultad para tragar
✅ dificultad para respirar
✅ dolor abdominal
✅ piel grasosa
✅ insomnio
✅ zumbido en oídos
✅ calambres
✅ sensibilidad al sonido
✅ erupciones
✅ desmayos
✅ dolor de cabeza
✅ picazón
✅ piel seca
✅ sudoración nocturna
✅ sed excesiva
✅ boca seca
✅ dolor al orinar
✅ sangrado
✅ dolor lumbar
✅ ictericia
✅ dolor articular
✅ tos con sangre
✅ pérdida de apetito
✅ micción frecuente
✅ rigidez matutina
✅ diarrea
✅ ojos rojos
✅ pérdida de equilibrio
✅ fatiga
✅ dolor muscular
✅ hormigueo
✅ moretones
✅ pérdida de memoria
✅ uñas frágiles
✅ mal aliento
✅ secreción nasal
✅ dolor de garganta
✅ convulsiones
✅ pérdida de peso
✅ inflamación
✅ dolor de pecho
✅ depresión
✅ entumecimiento
✅ aumento de peso
✅ debilidad muscular
✅ mareos
✅ escalofríos
✅ sensibilidad a la luz
✅ heces oscuras
✅ caída de cabello
✅ vómitos
✅ ansiedad
✅ manchas en la piel
✅ estornudos
✅ irritabilidad
✅ visión borrosa
✅ náuse

In [18]:
try:
    assert isinstance(sintomas_san_jose, list), "sintomas_san_jose debe ser una lista"
    assert len(sintomas_san_jose) >= 1, "La lista está vacía.\n  Pista: ¿Filtraste por p['provincia'] == 'San José'? ¿Tienes el for anidado sobre p['sintomas']?"
    print(f"✓ Desafío 2.C correcto — {len(sintomas_san_jose)} síntomas únicos en San José")
except NameError as e:
    print(f"Variable no definida: {e}")
    print("  ¿Ejecutaste la celda del desafío antes de esta?")
except AssertionError as e:
    print(f"✗ {e}")

✓ Desafío 2.C correcto — 64 síntomas únicos en San José


> 🤝 **Revisión en pareja — 5 min**
> 1. En 2.B, ¿pusieron el diccionario contador antes o dentro del `for` externo?
> 2. ¿Obtuvieron 3 000 000 en el assert? ¿De dónde viene ese número?
> 3. Comparen su solución de 2.C: ¿usaron el mismo orden de condiciones?

> ✋ **Checkpoint oral:**
> 1. ¿Cuántos ciclos `for` necesitamos para acceder a un síntoma individual?
> 2. ¿Por qué el assert dice 3 000 000 y no 1 000 000?
> 3. ¿Cómo modificarías 2.B para usar `Counter` en lugar del patrón manual?

---
## Fase 3 — `sorted()` con `key=lambda`

> **Objetivo:** ordenar diccionarios de conteos por valor usando una función anónima.

⏱ 30 minutos

> ⚠️ **Antes de continuar:** esta fase usa `conteo_enfermedades` (Fase 1) y `conteo_sintomas` (Fase 2).
> Asegúrate de haber ejecutado todas las celdas anteriores.

### Concepto — ¿Qué es una `lambda`?

Cuando llamamos a `sorted()`, necesitamos decirle **por qué criterio** ordenar.
Para eso usamos el parámetro `key=`, que recibe una función.

Podemos definir esa función de dos formas equivalentes:

```python
# Forma larga — función con nombre propio
def obtener_cantidad(par):
    return par[1]

sorted(conteo.items(), key=obtener_cantidad, reverse=True)

# Forma corta — lambda (función anónima de una sola línea)
sorted(conteo.items(), key=lambda par: par[1], reverse=True)
```

**Leemos la lambda en voz alta:** *"Para cada `par`, retorna `par[1]`"*.

La `lambda` no reemplaza al `def` — es una herramienta puntual para funciones simples
que no necesitan nombre propio porque se usan en un solo lugar.

> **Ritual de predicción:** `conteo_enfermedades.items()` retorna pares `(enfermedad, cantidad)`.
> Si `x = ("gripe", 4200)`, ¿qué retorna `lambda x: x[1]`?

In [19]:
# ── Mini-ejercicio — practica lambda antes del desafío ────────────────────
# Tenemos estos pares (ciudad, temperatura):
temperaturas = [("San José", 22), ("Liberia", 35), ("Cartago", 17), ("Puntarenas", 32)]

# Ordénalos de mayor a menor temperatura usando sorted() con key=lambda
# Reemplaza los ___ con el código correcto:
# ordenadas = sorted(temperaturas, key=lambda ___: ___[___], reverse=___)
# print(ordenadas)
ordenadas = sorted(temperaturas, key=lambda x: x[1], reverse=True)
print(f"Ciudad y temperaturas ordenadas: {ordenadas}")


Ciudad y temperaturas ordenadas: [('Liberia', 35), ('Puntarenas', 32), ('San José', 22), ('Cartago', 17)]


In [20]:
# ── Celda 3.0 — sorted() sobre pares (clave, valor) ────────────────────────
pares = list(conteo_enfermedades.items())
print(f"Tipo de cada elemento: {type(pares[0])}")   # <class 'tuple'>
print(f"Primer par: {pares[0]}")

ordenados = sorted(conteo_enfermedades.items(), key=lambda x: x[1], reverse=True)
for enfermedad, cantidad in ordenados:
    print(f"{enfermedad}: {cantidad}")

Tipo de cada elemento: <class 'tuple'>
Primer par: ('alzheimer', 24767)
alergia: 25472
otitis: 25297
esclerosis múltiple: 25273
parkinson: 25267
ansiedad generalizada: 25256
estrés: 25203
colitis: 25196
varicela: 25162
psoriasis: 25131
insuficiencia renal: 25127
trastorno bipolar: 25119
conjuntivitis: 25105
fibromialgia: 25096
epilepsia: 25083
hepatitis: 25074
hipertensión: 25062
arritmia: 25045
tiroiditis: 25044
diabetes: 25022
obesidad: 25012
anemia: 25009
faringitis: 24986
depresión mayor: 24981
migraña: 24956
lupus: 24943
hipertiroidismo: 24928
cistitis: 24916
lumbalgia: 24914
gripe: 24898
hipotiroidismo: 24890
gota: 24877
artritis: 24856
osteoporosis: 24831
bronquitis: 24803
alzheimer: 24767
asma: 24762
úlcera péptica: 24743
gastritis: 24673
dermatitis: 24615
sinusitis: 24606


> ⚠️ **Error frecuente — Olvidar `reverse=True`**
>
> Sin `reverse=True`, `sorted()` ordena de **menor a mayor** por defecto.
> Para rankings (más frecuente primero) siempre necesitamos `reverse=True`.
>
> ```python
> sorted(conteo.items(), key=lambda x: x[1])           # ❌ menor a mayor
> sorted(conteo.items(), key=lambda x: x[1], reverse=True)  # ✓ mayor a menor
> ```

### Desafío 3.A — Ranking de síntomas

⏱ 8 minutos | Individual

Ordena los síntomas de **más frecuente a menos frecuente** usando `sorted()` con `key=lambda`.
Guarda el resultado en `sintomas_ordenados` e imprime el ranking completo.

**Patrón (completa los espacios en blanco):**
```python
sintomas_ordenados = sorted(conteo_sintomas.items(), key=lambda ___: ___[___], reverse=___)
```

**Salida esperada (primeras líneas):**
```
(<síntoma_más_frecuente>, <cantidad>)
(<síntoma_segundo>, <cantidad>)
...
```

In [21]:
# Escribe tu código aquí
sintomas_ordenados = sorted(conteo_sintomas.items(), key=lambda x: x[1], reverse=True)
for sintoma, cantidad in sintomas_ordenados:
    print(f"Síntoma: {sintoma} y {cantidad:,} veces.")
    

Síntoma: desmayos y 47,470 veces.
Síntoma: sed excesiva y 47,286 veces.
Síntoma: moretones y 47,250 veces.
Síntoma: dolor de pecho y 47,220 veces.
Síntoma: hormigueo y 47,185 veces.
Síntoma: diarrea y 47,180 veces.
Síntoma: pérdida de apetito y 47,176 veces.
Síntoma: calambres y 47,174 veces.
Síntoma: piel seca y 47,159 veces.
Síntoma: mareos y 47,151 veces.
Síntoma: irritabilidad y 47,150 veces.
Síntoma: escalofríos y 47,139 veces.
Síntoma: confusión y 47,126 veces.
Síntoma: ansiedad y 47,126 veces.
Síntoma: dolor abdominal y 47,120 veces.
Síntoma: piel grasosa y 47,098 veces.
Síntoma: dolor de garganta y 47,095 veces.
Síntoma: dolor muscular y 47,090 veces.
Síntoma: sensibilidad al sonido y 47,079 veces.
Síntoma: estornudos y 47,055 veces.
Síntoma: heridas que no sanan y 47,033 veces.
Síntoma: tos y 47,026 veces.
Síntoma: aumento de peso y 47,022 veces.
Síntoma: náuseas y 46,983 veces.
Síntoma: insomnio y 46,968 veces.
Síntoma: boca seca y 46,968 veces.
Síntoma: esputo amarillo y 46,

### Desafío 3.B — Gráfico de barras ASCII

⏱ 10 minutos | Individual

Visualizamos los **10 síntomas más frecuentes** como un gráfico de barras directamente en la terminal, sin ninguna biblioteca externa.

**Formato esperado:**
```
fiebre                         ████████████████████████████████████████ (47,234)
dolor de cabeza                ██████████████████████████████████ (40,891)
tos                            ████████████████████████████ (34,102)
...
```

**Instrucciones:**
- Usa `sintomas_ordenados[:10]` de 3.A como fuente de datos.
- Construye la barra con `"█" * n` donde `n` es proporcional a la cantidad.
- **Normaliza** las barras para que la más larga ocupe exactamente **40 caracteres**: `escala = 40 / maximo`.
- Alinea los nombres a la izquierda con `:<30` para que las barras comiencen en la misma columna.
- Formatea las cantidades con separador de miles usando `{cantidad:,}`.

> **Pregunta de predicción antes de escribir el código:**
> Si el síntoma más frecuente tiene 47 000 ocurrencias y el segundo tiene 23 500, ¿cuántos `█` tendrá la barra del segundo?

In [22]:
# Escribe tu código aquí
top_10 = sintomas_ordenados[:10]
maximo = top_10[0][1]
escala = 50 / maximo
print("Top 10 síntomas más comunes:")
for sintoma, cantidad in top_10:
    barras = "█" * int(cantidad * escala)
    print(f"{sintoma:<20} | {barras} {cantidad:,} pacientes")   


Top 10 síntomas más comunes:
desmayos             | █████████████████████████████████████████████████ 47,470 pacientes
sed excesiva         | █████████████████████████████████████████████████ 47,286 pacientes
moretones            | █████████████████████████████████████████████████ 47,250 pacientes
dolor de pecho       | █████████████████████████████████████████████████ 47,220 pacientes
hormigueo            | █████████████████████████████████████████████████ 47,185 pacientes
diarrea              | █████████████████████████████████████████████████ 47,180 pacientes
pérdida de apetito   | █████████████████████████████████████████████████ 47,176 pacientes
calambres            | █████████████████████████████████████████████████ 47,174 pacientes
piel seca            | █████████████████████████████████████████████████ 47,159 pacientes
mareos               | █████████████████████████████████████████████████ 47,151 pacientes


> ✋ **Checkpoint oral:**
> 1. ¿Qué tipo de dato retorna `.items()`?
> 2. ¿Qué hace `lambda x: x[1]`?
> 3. ¿Cómo ordenarías de **menor a mayor**?
> 4. ¿Por qué necesitamos `escala = 40 / maximo` antes de dibujar las barras? ¿Qué pasaría si usáramos los valores directos?
>
> **Anticipo S10:** `df.sort_values("cantidad", ascending=False)` en Pandas reemplaza `sorted()` con `key=lambda` — misma idea, sintaxis más simple.

---
## Fase 4 — Integrador libre

⏱ 25 minutos

### Desafío final

Usando el **1 000 000 de registros** de la clínica, generamos un **reporte clínico** que responde al menos **2 preguntas**.

Elegimos según nuestro ritmo:

**Nivel básico**
- ¿Cuál es la edad promedio de los pacientes activos?
- ¿Cuántos pacientes hay por provincia?

**Nivel medio**
- ¿Qué provincia tiene el mayor promedio de visitas?
- ¿Cuál es la enfermedad más frecuente en cada provincia?

**Nivel avanzado**
- ¿Qué síntoma aparece asociado a más enfermedades distintas?
- ¿Cuál es el top 5 de pacientes con más visitas y qué enfermedad tienen?

---

**¿Por dónde empezar?**

```python
# BÁSICO — filtra con if, luego calcula
activos = [p for p in pacientes if p["activo"]]
promedio_edad = sum(p["edad"] for p in activos) / len(activos)

# MEDIO — agrupa con dict, luego procesa cada grupo
por_provincia = {}
for p in pacientes:
    prov = p["provincia"]
    if prov not in por_provincia:
        por_provincia[prov] = []
    por_provincia[prov].append(p["visitas"])

# AVANZADO — combina lo aprendido en Fases 1, 2 y 3
```

> 💡 **Bonus:** prueba la misma pregunta con `Counter` y con el patrón manual.
> ¿Notas diferencia de velocidad? Usa `%%time` para medirlo.

> 🤝 **Revisión en pareja:** comparte tu reporte y explica tu razonamiento.

In [23]:
# ── Tu reporte clínico — responde al menos 2 preguntas ────────────────────
# Usa los comentarios de la celda anterior como guía si te quedas en blanco.


---
## Cierre de sesión

**Una cosa nueva que aprendí hoy:**
_Escribe aquí..._

**Una conexión con algo de S01–S08:**
_Escribe aquí..._

---
## Glosario de la sesión

| Término | Definición |
|---------|------------|
| `dict.get(k, 0)` | Retorna el valor de la clave `k` o `0` si no existe. Evita `KeyError`. |
| `Counter` | Clase de `collections` que cuenta elementos de un iterable automáticamente. |
| `.most_common(n)` | Método de Counter que retorna los `n` elementos más frecuentes. |
| Bucle anidado | Un `for` dentro de otro `for`. Necesario cuando los datos tienen múltiples niveles. |
| `sorted(iterable, key=..., reverse=True)` | Retorna una lista ordenada. `key` define el criterio, `reverse=True` ordena descendente. |
| `lambda` | Función anónima de una línea. `lambda x: x[1]` retorna el segundo elemento de `x`. |
| `.items()` | Método de dict que retorna pares `(clave, valor)` como tuplas. |

---
## ¿Qué sigue? — Semana 10

En S10 damos un salto enorme: dejamos los bucles `for` manuales y conocemos **Pandas**,
la biblioteca que usan analistas y científicos de datos en todo el mundo.

- **Series y DataFrames:** la lista de diccionarios que hoy recorremos con `for` se convierte en una tabla manipulable en una sola línea.
- **Filtrado vectorizado:** el `for + if` que escribimos hoy se reemplaza por `df[df["columna"] == valor]`.
- **`value_counts()` y `groupby()`:** el `Counter` y los conteos que dominamos hoy son la base exacta de estas herramientas.

> Todo lo que aprendimos sobre diccionarios, conteos y ordenamiento fue preparación directa para Pandas.

---
*Fundamentos de Python — UMCA | Ing. Andrés Mena Abarca*